# SupeRR Selector: Data Understanding and Exploratory Data Analysis

I am building SupeRR Selector to identify which cricket players perform best when matches are on the line. In T20 cricket, aggregate statistics like total runs or wickets can be misleading because they do not account for match context. Scoring 30 runs when needing 12 off 30 balls is very different from scoring 30 runs when needing 14 an over with five wickets down.

Before building formulas for match pressure or player ratings, I need to thoroughly explore the raw data. This notebook covers the initial exploration of the six CSV datasets provided for the hackathon. My goal here is to check data types, verify table relationships, investigate anomalies, and establish baseline patterns across IPL and T20 International matches from 2022 to 2026.


## 1. Environment Setup and Display Configuration

I start by importing pandas, numpy, matplotlib, and seaborn. 

In ball-by-ball cricket data, each row has dozens of attributes including pitch types, shot descriptions, and multiple extra run categories. Pandas by default truncates columns and rows when printing tables, which can hide important context. I set the display options to show up to 35 columns and format floating-point numbers to three decimal places. I also configure a clean visual style for the plots generated later in this notebook.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visual settings for charts
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

# Display settings to inspect wide cricket tables without truncation
pd.set_option('display.max_columns', 35)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("Environment configured.")

## 2. Ingesting and Inspecting the Six Core Datasets

To start the audit, I need to load all six CSV files from the `Data Pack` directory into memory and check their shapes and memory footprints. 

Cricket data naturally spans multiple levels of hierarchy:
1. `competitions.csv`: High-level tournament catalog (IPL seasons and T20I series).
2. `teams.csv`: Team listings mapped to each competition.
3. `matches.csv`: Match-level context such as venues, toss outcomes, winners, and rain-adjusted targets.
4. `playing_eleven.csv`: Squad sheets, batting orders, captaincy, and impact player substitutions.
5. `competition_team_player_mapping.csv`: Master identity lookup linking player IDs with player names.
6. `ball_by_ball.csv`: The micro-level transaction log containing every delivery bowled.

I wrote a quick path check so this notebook works smoothly whether it is run from the project root or from inside the `notebooks` directory.

In [ ]:
# Resolve path dynamically whether executing from project root or notebooks folder
if os.path.exists("Data Pack"):
    data_dir = "Data Pack"
elif os.path.exists(os.path.join("..", "Data Pack")):
    data_dir = os.path.join("..", "Data Pack")
else:
    raise FileNotFoundError("Could not find the Data Pack directory.")

# Load all six CSV datasets
competitions = pd.read_csv(os.path.join(data_dir, "competitions.csv"))
teams = pd.read_csv(os.path.join(data_dir, "teams.csv"))
matches = pd.read_csv(os.path.join(data_dir, "matches.csv"))
playing_eleven = pd.read_csv(os.path.join(data_dir, "playing_eleven.csv"), low_memory=False)
player_mapping = pd.read_csv(os.path.join(data_dir, "competition_team_player_mapping.csv"))
ball_by_ball = pd.read_csv(os.path.join(data_dir, "ball_by_ball.csv"), low_memory=False)

datasets = {
    "competitions": competitions,
    "teams": teams,
    "matches": matches,
    "playing_eleven": playing_eleven,
    "player_mapping": player_mapping,
    "ball_by_ball": ball_by_ball
}

# Measure row counts, column counts, and memory usage
dataset_summary = []
for name, df in datasets.items():
    memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
    dataset_summary.append({
        "Dataset": name,
        "Rows": f"{len(df):,}",
        "Columns": df.shape[1],
        "Memory (MB)": f"{memory_mb:.2f} MB"
    })

pd.DataFrame(dataset_summary)

### Observations on Dataset Dimensions

The delivery table `ball_by_ball.csv` is our largest and most important file, containing 200,999 deliveries across 33 columns and taking about 68 MB in memory. `matches.csv` contains 883 match records, while `playing_eleven.csv` has 28,381 player-match appearances.

Because `ball_by_ball.csv` is the core grain of our analysis, performance matters. Later transformations such as computing cumulative match scores or rolling run rates should be vectorized using Pandas operations rather than slow row-by-row loops.

## 3. Schema and Data Types in the Delivery Table

Next, I want to inspect the column names and inferred data types in `ball_by_ball.csv`. 

If an ID column or flag is loaded with the wrong data type (for example, if IDs are parsed as floats because of nulls, or boolean flags are read as text strings), subsequent merges and aggregations can fail silently. Checking the schema early prevents subtle bugs down the line.

In [ ]:
# Inspect inferred data types and view a sample delivery record
print(ball_by_ball.dtypes)
print("\nSample delivery record:")
ball_by_ball.head(1).T

### Column Structure Insights

The data types match their cricket roles:
* Core identifiers (`match_id`, `striker_id`, `bowler_id`) are stored as clean 64-bit integers.
* Game counters (`innings`, `overid`, `over_ball_num`, `innings_ball_num`) are integers.
* Dismissal fields (`dismissal_type_id`, `dismissed_player_id`) are stored as `float64` because they contain missing values on balls where no wicket fell.
* Boolean indicators (`is_four`, `is_six`, `is_wicket`) are properly loaded as boolean types (`True`/`False`).
* Descriptive columns (`pace_or_spin`, `stroke`, `delivery_style`) are stored as strings.

This means we can rely on standard integer joins for match and player IDs, and direct boolean masking for boundaries and wickets.

## 4. Validating Primary Keys and Relational Integrity

Before doing any multi-table analysis, I need to verify that primary keys are unique and that foreign keys point to valid records. 

If a match ID in `ball_by_ball.csv` does not exist in `matches.csv`, or if delivery records have duplicate entries for the same ball, our aggregated statistics will be corrupted. I test:
1. Whether `comp_id` uniquely identifies tournaments in `competitions.csv`.
2. Whether `match_id` uniquely identifies matches in `matches.csv`.
3. What combination of columns uniquely identifies a delivery in `ball_by_ball.csv`.
4. Whether all matches in `ball_by_ball.csv` exist in `matches.csv`.

In [ ]:
# Verify primary key candidate uniqueness
print("competitions comp_id is unique:", competitions['comp_id'].is_unique)
print("matches match_id is unique:", matches['match_id'].is_unique)

# Test candidate keys for deliveries
ball_duplicate_check_1 = ball_by_ball.duplicated(subset=['match_id', 'innings', 'overid', 'over_ball_num']).sum()
ball_duplicate_check_2 = ball_by_ball.duplicated(subset=['match_id', 'innings', 'innings_ball_num']).sum()

print("Duplicates on (match_id, innings, overid, over_ball_num):", ball_duplicate_check_1)
print("Duplicates on (match_id, innings, innings_ball_num):", ball_duplicate_check_2)

# Test referential integrity
all_matches_valid = ball_by_ball['match_id'].isin(matches['match_id']).all()
print("All ball_by_ball match IDs exist in matches.csv:", all_matches_valid)

### Verification Findings

Both candidate composite keys for `ball_by_ball.csv` have zero duplicates across all 200,999 rows. This confirms that `(match_id, innings, overid, over_ball_num)` is a completely reliable primary key. 

Furthermore, every match present in the delivery records has a matching record in `matches.csv`. The relational structure is solid and safe for joins.

## 5. Investigating the 16 Matches with No Delivery Data

While checking match counts, I noticed an interesting discrepancy: `matches.csv` contains 883 matches, but `ball_by_ball.csv` only contains deliveries for 867 unique matches. Exactly 16 matches appeared to have missing data.

I wanted to find out why these 16 matches were missing from the ball-by-ball file before assuming this was a data pipeline error. I filtered `matches.csv` to find the 16 match IDs and examined their results.

In [ ]:
# Identify match IDs that exist in matches.csv but not in ball_by_ball.csv
all_match_ids = set(matches['match_id'].unique())
ball_match_ids = set(ball_by_ball['match_id'].unique())
missing_match_ids = all_match_ids - ball_match_ids

print(f"Total matches in matches.csv: {len(all_match_ids)}")
print(f"Matches with delivery data: {len(ball_match_ids)}")
print(f"Matches without delivery data: {len(missing_match_ids)}")

# Inspect the result field for these 16 matches
missing_matches_df = matches[matches['match_id'].isin(missing_match_ids)][
    ['match_id', 'match_date', 'stage', 'home_team', 'away_team', 'result']
]
missing_matches_df.head(10)

### Explanation of the Missing Matches

Looking at the `result` column, every single one of these 16 matches has `result == 'Match Abandoned'`. These fixtures were rained out without a toss or ball bowled (such as KKR vs GT in IPL 2024).

This confirms that the data is not corrupted. These matches simply have no deliveries because no play was possible. When evaluating player appearance rates or team statistics later, I need to account for these abandoned fixtures so they do not skew participation metrics.

## 6. Missing Value Analysis in Cricket Data

Next, I analyzed missing values across the core datasets. In general data cleaning, missing values often indicate faulty collection or omitted responses. In sports data, however, nulls are frequently meaningful representations of the game itself. 

I wrote a small helper function to inspect null counts and percentages across `ball_by_ball.csv`, `matches.csv`, and `playing_eleven.csv`.

In [ ]:
def audit_missing_values(df, name):
    null_counts = df.isnull().sum()
    null_percentages = (null_counts / len(df)) * 100
    null_summary = pd.DataFrame({
        "Missing Count": null_counts,
        "Percentage (%)": null_percentages
    })
    null_summary = null_summary[null_summary["Missing Count"] > 0].sort_values(
        by="Missing Count", ascending=False
    )
    print(f"=== Missing Values in {name} ===")
    if null_summary.empty:
        print("No missing values found.")
    else:
        print(null_summary)
    print()

audit_missing_values(ball_by_ball, "ball_by_ball.csv")
audit_missing_values(matches, "matches.csv")
audit_missing_values(playing_eleven, "playing_eleven.csv")

### Understanding the Null Values

The missing values reflect how cricket works:
* `dismissal_type` and `dismissal_type_id` are missing in 94.59% of records. This is because wickets only fall on about 5.4% of deliveries.
* `extras_type` is missing in 94.11% of deliveries because the vast majority of deliveries are legal balls off the bat without wides, no-balls, leg-byes, or byes.
* In `matches.csv`, `revised_target` and revised overs are missing in over 97% of matches because rain interruptions requiring DLS adjustments are relatively infrequent.
* In `playing_eleven.csv`, roughly 30% of batting orders are unassigned, which happens when tailenders are listed on the team sheet but do not bat during the match.

None of these columns should be carelessly dropped or filled with dummy numbers. We will only use them when their specific conditions apply.

## 7. Numerical Distributions and Cricket Rule Invariants

I wanted to check whether the numerical values in `ball_by_ball.csv` obey standard cricket rules. Specifically:
1. Are runs scored off the bat within the valid range of 0 to 6?
2. Do the `is_four` and `is_six` boolean flags match the recorded runs?
3. What is the maximum total runs conceded on a single delivery when accounting for extras?

In [ ]:
# Check run distribution off the bat
print("Runs off the bat frequency:")
print(ball_by_ball['runs'].value_counts().sort_index())

# Compare boundary counts
runs_four_count = (ball_by_ball['runs'] == 4).sum()
flag_four_count = ball_by_ball['is_four'].sum()
runs_six_count = (ball_by_ball['runs'] == 6).sum()
flag_six_count = ball_by_ball['is_six'].sum()

print(f"\nRuns == 4: {runs_four_count:,} | is_four == True: {flag_four_count:,}")
print(f"Runs == 6: {runs_six_count:,} | is_six == True: {flag_six_count:,}")

# Check total runs scored on a delivery (runs off bat + all extras)
total_runs_on_ball = ball_by_ball['runs'] + ball_by_ball['extras1_runs'] + ball_by_ball['extras_2_runs']
print(f"Maximum total runs on a single ball: {total_runs_on_ball.max()}")
print(f"Minimum total runs on a single ball: {total_runs_on_ball.min()}")

### Invariant Check Results

The distributions align with T20 match realities:
* Dots (0 runs): 79,387 balls (39.50% of all deliveries).
* Singles (1 run): 72,267 balls (35.95%).
* Twos: 12,547 balls (6.24%).
* Threes: 656 balls (0.33%).
* Fours: 23,303 balls (11.59%).
* Fives: 25 balls (0.01%, typically overthrows).
* Sixes: 12,814 balls (6.38%).

The boundary flags match the run values closely (with tiny variations where boundary all-run events occur). The maximum total runs on a single ball was 7 (a no-ball hit for six), and the minimum was 0. The dot ball percentage of nearly 40% shows how important strike rotation is in T20 cricket. Later, when evaluating pressure performance, a batter's ability to minimize dot balls under pressure will be a major evaluation metric.

## 8. Tactical Distributions: Pace vs Spin, Dismissals, and Toss Decisions

I also wanted to examine three tactical elements in modern T20 cricket:
1. How often teams use pace bowling versus spin bowling.
2. The most common ways batters lose their wickets.
3. How captains behave after winning the toss.

Understanding these distributions provides essential context. For example, if pacers bowl most of the death overs, their raw economy rates will naturally look higher than those of spinners who bowl in the middle overs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Pace vs Spin breakdown
pace_spin_counts = ball_by_ball['pace_or_spin'].value_counts()
axes[0].bar(pace_spin_counts.index, pace_spin_counts.values, color=['#2b5c8f', '#d97724'])
axes[0].set_title("Deliveries by Bowling Style")
axes[0].set_ylabel("Ball Count")
for idx, count in enumerate(pace_spin_counts.values):
    axes[0].text(idx, count * 0.9, f"{count:,}\n({count/len(ball_by_ball):.1%})", 
                 ha='center', color='white', fontweight='bold')

# 2. Top dismissal methods
top_dismissal_types = ball_by_ball['dismissal_type'].value_counts().head(5)
axes[1].barh(top_dismissal_types.index, top_dismissal_types.values, color='#2e7d32')
axes[1].set_title("Top 5 Dismissal Methods")
axes[1].set_xlabel("Number of Wickets")
axes[1].invert_yaxis()

# 3. Toss decisions
toss_decisions = matches['toss_opted'].value_counts()
axes[2].pie(toss_decisions.values, labels=toss_decisions.index, autopct='%1.1f%%', 
            colors=['#c62828', '#6a1b9a'], startangle=140)
axes[2].set_title("Toss Decisions (All Matches)")

plt.tight_layout()
plt.show()

### Tactical Observations

* **Pace vs Spin:** Pace accounts for 57.7% of all deliveries (115,892 balls) while Spin accounts for 42.3% (85,107 balls). This fits the modern T20 pattern where fast bowlers handle both the opening powerplay and the closing death overs, while spinners primarily manage the middle overs.
* **Dismissal Modes:** 'Caught' is the dominant dismissal mode, accounting for over 60% of all wickets. Bowled, LBW, Run Out, and Stumped follow.
* **Toss Bias:** Teams chose to field first in 68.3% of matches, batting first in only 31.7%. Chasing targets with known run rates remains the dominant strategy in modern T20 cricket, highlighting why second-innings chases require specialized required-run-rate modeling.

## 9. Scoring Rates and Wicket Hazards by Match Phase

T20 innings naturally separate into three tactical phases:
* **Powerplay (Overs 1 to 6):** 30-yard circle fielding restrictions with only 2 outfielders.
* **Middle Overs (Overs 7 to 15):** 5 fielders outside the circle, with spinners slowing the tempo.
* **Death Overs (Overs 16 to 20):** All-out aggression, high boundary demand, and specialized death bowling.

To quantify these phases, I aggregated the total runs scored and total wickets fallen for each of the 20 overs across all matches.

In [ ]:
# Aggregate runs and wickets per over
over_summary = ball_by_ball.groupby('overid').agg(
    total_balls=('runs', 'count'),
    total_runs=('runs', 'sum'),
    total_wickets=('is_wicket', 'sum')
).reset_index()

# Focus on the standard 20 overs
standard_overs = over_summary[over_summary['overid'] <= 20].copy()
standard_overs['run_rate'] = standard_overs['total_runs'] / (standard_overs['total_balls'] / 6)
standard_overs['wicket_probability_pct'] = (standard_overs['total_wickets'] / standard_overs['total_balls']) * 100

fig, ax1 = plt.subplots(figsize=(12, 5))

# Plot run rate
line_color = '#1f77b4'
ax1.set_xlabel('Over Number', fontsize=12)
ax1.set_ylabel('Run Rate (Runs per 6 Balls)', color=line_color, fontsize=12)
ax1.plot(standard_overs['overid'], standard_overs['run_rate'], marker='o', 
         linewidth=2.5, color=line_color, label='Run Rate')
ax1.tick_params(axis='y', labelcolor=line_color)
ax1.set_xticks(range(1, 21))

# Highlight the three phases
ax1.axvspan(1, 6, alpha=0.15, color='green', label='Powerplay (Overs 1-6)')
ax1.axvspan(7, 15, alpha=0.10, color='gold', label='Middle (Overs 7-15)')
ax1.axvspan(16, 20, alpha=0.15, color='crimson', label='Death (Overs 16-20)')

# Twin axis for wicket probability
ax2 = ax1.twinx()
wicket_color = '#d62728'
ax2.set_ylabel('Wicket Probability per Ball (%)', color=wicket_color, fontsize=12)
ax2.plot(standard_overs['overid'], standard_overs['wicket_probability_pct'], marker='s', 
         linewidth=2, linestyle='--', color=wicket_color, label='Wicket %')
ax2.tick_params(axis='y', labelcolor=wicket_color)
ax2.grid(False)

plt.title("Run Rate and Wicket Probability by Over (Powerplay vs Middle vs Death)", fontsize=13, pad=15)
ax1.legend(loc='upper left')
ax2.legend(loc='lower right')
plt.show()

### Phase Dynamics Insights

The chart reveals distinct tactical phases:
* **Powerplay (Overs 1 to 6):** Scoring begins briskly at roughly 7.8 to 8.2 runs per over. Wicket probability is elevated in overs 1 and 2 (~6.5%) as openers negotiate the new ball.
* **Middle Overs (Overs 7 to 15):** Once the fielding restrictions end, the run rate drops to between 7.2 and 7.8 runs per over while wicket probability hovers around 4.5% to 5.2%.
* **Death Overs (Overs 16 to 20):** In the final five overs, run rates climb sharply towards 10.5+ runs per over, and wicket probability peaks at nearly 8.5% per delivery in over 20.

This confirms why match phase must be part of our pressure definition: death overs are inherently high-leverage environments for both batters and bowlers.

## 10. Preliminary Player Volume Baselines

Before evaluating player performance under pressure, I wanted to find the overall volume leaders in runs and bowler-credited wickets across the entire dataset. 

When counting wickets for bowlers, it is essential to exclude run outs, retired out, and retired hurt, because those wickets belong to the fielding side rather than the bowler's individual tally.

In [ ]:
# 1. Top 5 run scorers across all competitions
top_batters = ball_by_ball.groupby('striker_name')['runs'].sum().sort_values(ascending=False).head(5)
print("Top 5 run scorers:")
print(top_batters)

# 2. Top 5 wicket takers credited to the bowler
bowler_wickets = ball_by_ball[
    (ball_by_ball['is_wicket'] == True) & 
    (~ball_by_ball['dismissal_type'].isin(['run out', 'retired out', 'retired hurt']))
]
top_bowlers = bowler_wickets.groupby('bowler_name')['is_wicket'].count().sort_values(ascending=False).head(5)
print("\nTop 5 wicket takers (bowler-credited wickets):")
print(top_bowlers)

# 3. Matches that went to a Super Over
super_over_matches = ball_by_ball[ball_by_ball['innings'] > 2]['match_id'].nunique()
print(f"\nMatches decided by a Super Over: {super_over_matches}")

### Volume Leader Takeaways

The volume leaders from 2022 to 2026 are:
* **Batters:** Suryakumar Yadav (4,595 runs), Jos Buttler (4,396 runs), Abhishek Sharma (4,332 runs), Shubman Gill (4,050 runs), and Tilak Varma (3,596 runs).
* **Bowlers:** Arshdeep Singh (211 wickets), Rashid Khan (159 wickets), Varun Chakravarthy (138 wickets), Wanindu Hasaranga (135 wickets), and Yuzvendra Chahal (131 wickets).
* Exactly 6 matches were tied at the end of 40 overs and required a Super Over.

These numbers give us our overall baseline. However, high volume alone does not tell us who performs when the match is on the line. In the upcoming modules, we will separate these overall numbers from their clutch equivalents.

## 11. Module 1 Summary and Next Steps

This initial exploration established several key facts about our data:
1. **Grain of Analysis:** Every event traces back to the delivery level `(match_id, innings, overid, over_ball_num)`, which is uniquely keyed across all 200,999 records.
2. **Data Cleanliness:** The relational structure is intact. The 16 matches without ball records were verified as rain abandonments rather than pipeline drops.
3. **Domain Realities:** Missing values in dismissals and extras are domain-valid events. Nearly 40% of deliveries are dot balls, and death overs see run rates jump to 10.5+ while wicket hazards climb to 8.5%.
4. **The Core Analytical Question:** Raw averages cannot evaluate clutch performance. To know whether a player delivers under pressure, we need to know the state of the match at the moment each ball was bowled.

---

### Plan for Module 2: Dynamic Match-State Reconstruction
In Module 2, I will build `src/match_state.py` and `notebooks/02_match_state.ipynb` to reconstruct the live context of every delivery:
* Cumulative runs and wickets fallen before each ball.
* Balls remaining and wickets in hand.
* Target score and runs needed for second-innings chases.
* Dynamic Current Run Rate (CRR) and Required Run Rate (RRR) on every delivery.
* Match phase labels (Powerplay, Middle, Death).